# Demo 03 - NYC Taxi Zone Lookup

Notebook này dùng để demo bảng tra cứu zone. Dataset này nhỏ nhưng rất quan trọng vì nó map `LocationID` trong taxi trips sang `Zone` và `Borough`.

## 4 phần nên chụp vào slide

1. Dataset overview
2. Schema / thuộc tính chính
3. Sample records
4. Borough summary

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebooks.utils.spark_session import get_spark
from pyspark.sql.functions import col, count

spark = get_spark("MetroPulse Demo - Taxi Zone Lookup")
spark.sparkContext.setLogLevel("WARN")
spark.conf.get("spark.sql.session.timeZone")

## Load taxi zone lookup

In [ ]:
zone_path = PROJECT_ROOT / "data/taxi_zone_lookup.csv"

zones = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(zone_path))
    .cache()
)

zone_path

## 1. Dataset overview

In [ ]:
overview = spark.createDataFrame(
    [
        ("Source", "NYC Taxi & Limousine Commission"),
        ("Scope", "NYC taxi zone lookup"),
        ("Expected zones", "263"),
        ("Role", "Map LocationID to Zone and Borough"),
        ("Local file", str(zone_path.relative_to(PROJECT_ROOT))),
    ],
    ["property", "value"],
)
overview.show(truncate=False)

## 2. Các thuộc tính chính

In [ ]:
attributes = spark.createDataFrame(
    [
        ("LocationID", "integer", "Khóa join với PULocationID / DOLocationID"),
        ("Borough", "string", "Quận / khu vực hành chính"),
        ("Zone", "string", "Tên taxi zone"),
        ("service_zone", "string", "Nhóm dịch vụ taxi"),
    ],
    ["column", "type", "meaning"],
)
attributes.show(truncate=False)
zones.printSchema()

## 3. Sample records

In [ ]:
zones.select("LocationID", "Borough", "Zone", "service_zone").show(10, truncate=False)

## 4. Borough summary

In [ ]:
zones.groupBy("Borough").agg(
    count("*").alias("zone_count")
).orderBy(col("zone_count").desc()).show(truncate=False)